# Stage 5: analyze

**Goal.** Answer the project's questions from two tables, the per-tile results and the proxies:

1. Do landscape proxies predict where the model is wrong? (RQ1)
2. Do they add anything to the model's own uncertainty? (RQ2)

**Command-line equivalent:** `p037 analyze --config <config> [--tiles <results table>]`

**Code:** `src/p037/analyze.py` (the stage), `src/p037/ood.py` (the numerics), `src/p037/plots.py`

This notebook walks through the steps one at a time, then runs the stage itself, which writes the
same things as tables, figures and `report.md`.

In [ ]:
import os
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from p037 import plots                      # sets the figure style used in the report
from p037.config import REPO_ROOT, load_config
from p037.datasets import get_dataset

# Default: the one real result that exists so far, the zero-shot run of PR #4 (its per-tile table is in
# the repository, so this notebook runs on a laptop). For your own run, set CONFIG to its config and
# TILES to None, which means <run folder>/tiles.csv.
CONFIG = os.environ.get("P037_CONFIG", "analysis/p037/configs/zeroshot_usf.yaml")
TILES = os.environ.get("P037_TILES", "analysis/usf_results.csv" if "zeroshot" in CONFIG else "") or None
cfg = load_config(REPO_ROOT / CONFIG)
ds = get_dataset(cfg)
DRY_RUN = cfg.dataset == "synthetic"
print(f"experiment {cfg.name!r} on dataset {cfg.dataset!r}")
print("dataset files:", cfg.dataset_dir)
tiles_csv = REPO_ROOT / TILES if TILES else cfg.out / "tiles.csv"
print("per-tile results:", tiles_csv)
print("training reference:", cfg.reference_set)

## 5.1 Join results and proxies, define "error"

`prepare` joins the two tables on `tile_id`, places every tile in the training distribution
(notebook 04) and adds three error measures:

| Measure | Definition | Defined for |
|---|---|---|
| `err_rate` | wrong pixels / scored pixels | every tile |
| `f1_loss` | 1 - F1 | tiles with at least `min_positive_px` flood pixels in the label |
| `miss_rate` | missed flood pixels / flood pixels | the same tiles |

Why three: F1 is undefined on a tile without flood, and most tiles have none. `err_rate` covers
every tile but is small wherever there is little flood to get wrong, so it partly measures *how
much flood a tile has*. That confound is handled explicitly below.

In [ ]:
from p037 import ood
from p037.analyze import UNCERTAINTY, prepare

df, ref, columns, info = prepare(cfg, tiles_csv)
print(info)
print(f"flooded tiles (>= {cfg.min_positive_px} flood px): {int(df['flooded'].sum())} of {len(df)}")
df[["tile_id", "event", "err_rate", "f1_loss", "miss_rate", "gt_frac", "log_mc_var"]].describe().round(3)

## 5.2 Does the model's own uncertainty track its error?

Rank correlation (Spearman) between tile error rate and MC-dropout variance, per scene.

The second column removes what the amount of flood in the tile explains. A tile with more flood
has more pixels to get wrong *and* more uncertain boundary pixels, which produces a correlation
by itself. The partial correlation asks what is left once that is accounted for.

In [ ]:
rows = []
for event, g in [*df.groupby("event"), ("all", df)]:
    rows.append({"event": event, "tiles": len(g),
                 "error ~ uncertainty": ood.spearman(g["err_rate"], g["log_mc_var"]),
                 "same, flood share removed": ood.partial_spearman(g["err_rate"], g["log_mc_var"], g["gt_frac"])})
pd.DataFrame(rows).set_index("event").round(2)

In [ ]:
events = sorted(df["event"].unique())
fig, axes = plt.subplots(1, len(events), figsize=(3.1 * len(events), 3), sharex=True, sharey=True, squeeze=False)
for ax, event in zip(axes[0], events):
    g = df[df["event"] == event]
    ax.scatter(g["log_mc_var"], g["err_rate"], s=4, alpha=0.3, color=plots.BLUE, linewidths=0)
    ax.set_title(plots.nice_event(event), loc="left"); ax.set_xlabel("log10 MC-dropout variance")
axes[0, 0].set_ylabel("tile error rate")
fig.tight_layout()

## 5.3 Does error depend on where a tile sits in the training distribution?

One proxy at a time. `position` is signed (is "more of this" worse?), `distance` is unsigned (is
"unlike training" worse?). Values near zero mean the proxy alone says little about error.

In [ ]:
rows = []
for p in columns:
    rows.append({"proxy": plots.nice_proxy(p),
                 "error ~ position": ood.spearman(df["err_rate"], df[f"pct__{p}"]),
                 "error ~ position, flood share removed": ood.partial_spearman(df["err_rate"], df[f"pct__{p}"], df["gt_frac"]),
                 "error ~ distance": ood.spearman(df["err_rate"], df[f"dist__{p}"])})
pd.DataFrame(rows).set_index("proxy").round(2)

## 5.4 The weighted OOD score

The supervisor's method: regress per-tile error on the per-proxy scores. The coefficients are
importance weights, and the weighted sum is a tile's OOD score.

Features are standardised, so a weight is the change in error rate for one training standard
deviation of that proxy's score, and weights are comparable between proxies.

The intervals come from refitting on resampled **blocks** of neighbouring tiles. Neighbouring
tiles share a river or a city and are not independent, so resampling single tiles would make the
weights look more certain than they are. A weight whose interval spans zero is not established.

In [ ]:
kept, dropped = ood.prune_collinear(ref, columns, cfg.max_proxy_corr)
dist_cols = [f"dist__{p}" for p in kept]
d = df.dropna(subset=["err_rate", *dist_cols]).reset_index(drop=True)
X, y = d[dist_cols].to_numpy(float), d["err_rate"].to_numpy(float)

model = ood.Linear().fit(X, y)
boot = ood.bootstrap_weights(X, y, ood.spatial_blocks(d), cfg.n_bootstrap, cfg.seed)
lo, hi = np.percentile(boot, [2.5, 97.5], axis=0)
weights = pd.DataFrame({"weight": model.weights, "lo": lo, "hi": hi}, index=[plots.nice_proxy(p) for p in kept])
print(f"dropped as collinear: {list(dropped)}")
print(f"in-sample R2 {ood.r2(y, model.predict(X)):.3f}, rank correlation {ood.spearman(model.predict(X), y):.2f}")

fig, ax = plt.subplots(figsize=(6, 0.45 * len(kept) + 1.2))
ypos = np.arange(len(kept))[::-1]
ax.hlines(ypos, weights["lo"], weights["hi"], color=plots.BLUE, lw=2)
ax.plot(weights["weight"], ypos, "o", color=plots.BLUE)
ax.axvline(0, color=plots.AXIS, lw=1)
ax.set_yticks(ypos, weights.index); ax.set_xlabel("weight: change in tile error rate per training SD of distance")
ax.set_title("Weights of the OOD score (95 % block-bootstrap interval)", loc="left")
fig.tight_layout()
weights.round(4)

The fit above is **in-sample**: the same tiles choose the weights and judge them. It shows which
proxies carry weight, and says nothing about whether the score would work on a new flood.

## 5.5 The honest test: an event the predictor has never seen

Fit on all events but one, predict the error of every tile of the one left out, repeat for each
event. The score is the rank correlation between predicted and observed error on the unseen
event, because a ranking is what a reviewer deciding where to look needs.

Three predictors are compared:

- **uncertainty**: MC-dropout variance, entropy, predicted flood share. Needs only the model.
- **proxies**: position and distance on the kept proxies. Needs only a map of the area.
- **uncertainty + proxies**: both.

RQ1 is whether *proxies* is clearly above zero. RQ2 is whether *both* beats *uncertainty*.

In [ ]:
pos_cols = [f"pos__{p}" for p in kept]
feature_sets = {"uncertainty": UNCERTAINTY, "proxies": pos_cols + dist_cols,
                "uncertainty+proxies": UNCERTAINTY + pos_cols + dist_cols}
if df["event_group"].nunique() >= 2:
    held, preds = ood.held_out_event_test(df, feature_sets, "err_rate")
    table = held.pivot(index="held_out", columns="model", values="spearman")[list(feature_sets)]
    display(table.round(2))
    fig, ax = plt.subplots(figsize=(6.5, 0.8 * len(table) + 1.3))
    for k, (name, (colour, label)) in enumerate(plots.MODEL_STYLE.items()):
        ax.plot(table[name], np.arange(len(table)) + (k - 1) * 0.2, "o", color=colour, label=label)
    ax.axvline(0, color=plots.AXIS, lw=1)
    ax.set_yticks(range(len(table)), [plots.nice_event(e) for e in table.index]); ax.invert_yaxis()
    ax.set_xlabel("rank correlation of predicted and observed tile error on the held-out event")
    ax.legend(loc="upper center", bbox_to_anchor=(0.5, -0.25), ncol=3)
    ax.set_title("Predicting tile error on an event never seen by the predictor", loc="left")
else:
    print("only one event: the held-out test needs at least two")

How to read it, and its limits:

- With three or four events, each number comes from a fit on the other two or three. Treat a
  single value as an indication. A pattern that repeats across events and across experiments is
  the evidence.
- A scene where **nothing** predicts the error is itself a finding: its failure has a cause that
  neither the model nor these proxies see.

## 5.6 The whole stage in one call

`run_analysis` computes everything above (plus the triage curves and per-scene maps), and writes
the tables, the figures and `report.md`. This is what `p037 analyze` runs.

In [ ]:
from IPython.display import Image, Markdown

from p037.analyze import run_analysis

out_dir = cfg.out / "analysis_notebook"
report = run_analysis(cfg, tiles_csv, out_dir)
print("written to", out_dir)
print("tables: ", sorted(p.name for p in out_dir.glob("*.csv*")))
print("figures:", sorted(p.name for p in (out_dir / "figures").glob("*.png")))

In [ ]:
display(Image(filename=str(out_dir / "figures" / "placement.png"), width=900))

In [ ]:
triage = out_dir / "figures" / "triage.png"
if triage.exists():
    display(Image(filename=str(triage), width=900))
    display(pd.read_csv(out_dir / "triage_top20.csv").round(2))

**Triage** turns the ranking into something an analyst would feel: if only the top 20 % of
tiles (by predicted error) are checked by a person, what share of all wrongly mapped pixels do
they see? `perfect_order` is the best any ranking could do.

## 5.7 Comparing two runs on the same tiles

Once Experiment B has run, this puts it next to the zero-shot result, on exactly the tiles both
tables contain:

In [ ]:
from p037.analyze import compare_runs

exp_b = Path(cfg.data_root) / "runs" / "exp_b_usf" / "tiles.csv"
zero_shot = REPO_ROOT / "analysis" / "usf_results.csv"
if exp_b.exists() and zero_shot.exists():
    display(compare_runs({"zero-shot": zero_shot, "Experiment B": exp_b}, "flood", cfg.score_class).round(3))
else:
    print("Experiment B has no tiles.csv yet:", exp_b)

## What goes in the report

- `placement.png`: proxy distributions, training against test (requested by the supervisor).
- `error_vs_confidence.png`, `error_vs_proxy.png`: the two "error against ..." plots.
- `weights_*.png`: the OOD weights with intervals.
- `held_out_*.png` and `triage.png`: the test of whether any of it transfers.

`report.md` holds the numbers behind each figure. It deliberately contains no interpretation:
that is yours to write.